# Práctica 3: Soluciones de los ejercicios propuestos
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

Los ejercicios que se plantean a continuación tienen como objetivo el practicar con la biblioteca [NLTK](https://www.nltk.org) de Python.

### Ejercicio 1

El objetivo de este ejercicio es entrenar y evaluar el rendimiento de un filtro de correo electrónico no deseado. Para ello se usará el corpus Enron-Spam, pero no se proporcionará un vocabulario fijo, sino que este deberá aprenderse a partir de los mensajes de entrenamiento. Con el objetivo de homogeneizar el vocabulario aprendido y de mejorar el rendimiento del filtro construido, se pedirá que se apliquen distintas técnicas de preprocesado.

En todos los apartados de este ejercicio se deberá realizar lo siguiente:

* Construir el filtro como una tubería de scikit-learn que concatene un vectorizador tf-idf y un modelo $k$NN clasificador con 5 vecinos y que use la métrica del coseno.
* Definir una función `procesa_mensaje` que, dado el contenido en bruto de un mensaje, aplique todos los pasos de procesamiento pedidos hasta obtener la lista de tókenes correspondiente. Esta función se deberá proporcionar como argumento `analyzer` del vectorizador tf-idf.
* Entrenar el filtro con el corpus de entrenamiento.
* Calcular la sensibilidad del filtro sobre el corpus de prueba.

In [1]:
from email import parser
from email import policy

In [2]:
analizador_mensaje = parser.Parser(policy=policy.default)

In [3]:
from pathlib import Path

In [4]:
carpeta_Enron_Spam = Path('Filtro antispam/Enron-Spam/')
carpeta_entrenamiento = carpeta_Enron_Spam / 'train'
carpeta_prueba = carpeta_Enron_Spam / 'test'

contenidos_mensajes_entrenamiento = []
clases_mensajes_entrenamiento = []
for ruta_mensaje in (carpeta_entrenamiento / 'legítimo').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_entrenamiento.append(mensaje.get_content())
            clases_mensajes_entrenamiento.append(0)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass
for ruta_mensaje in (carpeta_entrenamiento / 'no_deseado').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_entrenamiento.append(mensaje.get_content())
            clases_mensajes_entrenamiento.append(1)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

contenidos_mensajes_prueba = []
clases_mensajes_prueba = []
for ruta_mensaje in (carpeta_prueba / 'legítimo').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_prueba.append(mensaje.get_content())
            clases_mensajes_prueba.append(0)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass
for ruta_mensaje in (carpeta_prueba / 'no_deseado').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_prueba.append(mensaje.get_content())
            clases_mensajes_prueba.append(1)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

#### Apartado 0

En este apartado se pide procesar los mensajes realizando los siguientes 3 pasos:

* Extraer el contenido de texto de los mensajes en formato HTML. Para ello hacer uso de la biblioteca [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/).
* Dividir el contenido de los mensajes en secuencias de tókenes mediante el tokenizador de NLTK.
* Eliminar de los tókenes los caracteres no alfanuméricos (y eliminar por completo aquellos tókenes que no contengan caracteres alfanuméricos).

In [5]:
contenidos_mensajes_entrenamiento[-21]

'OurRef:CBN/GO/0X012/05 \nDATE:21st JULY 2005\nTEL:234-1-470-7915\nEMAIL ADDRESS:mallamyusuf@centbanks.org \n\nDear Good Friend \n\nAfter a serious thought, I decided to reach you directly and personally\nbecause I do not have anything against you, but your Nigerian partners. \n\nI am the director of wire transfer/telex department of the central bank of\nNigeria, some time in the past your Nigerian partners approached me through a\nfriend of mine who works with one of the ministries here and requested that I\nassist them conclude a money transfer deal and we all agreed. \n\nAccording to them, they wanted to use this strategy to transfer a huge amount\nof us dollars which they accumulated through inflated contract awards and\nthemoney has been floating in the (c.b.n) since the original beneficiary has\nbeen fully paid, so they wanted to use your account to transfer the surplus\nout of Nigeria. We agreed that once i do this, they would give me\nUS$100,000.00 and give me another us100, 00

In [6]:
from bs4 import BeautifulSoup

In [7]:
def elimina_html(contenido):
    return BeautifulSoup(contenido).get_text()

In [8]:
elimina_html(contenidos_mensajes_entrenamiento[-21])

'OurRef:CBN/GO/0X012/05 \nDATE:21st JULY 2005\nTEL:234-1-470-7915\nEMAIL ADDRESS:mallamyusuf@centbanks.org \n\nDear Good Friend \n\nAfter a serious thought, I decided to reach you directly and personally\nbecause I do not have anything against you, but your Nigerian partners. \n\nI am the director of wire transfer/telex department of the central bank of\nNigeria, some time in the past your Nigerian partners approached me through a\nfriend of mine who works with one of the ministries here and requested that I\nassist them conclude a money transfer deal and we all agreed. \n\nAccording to them, they wanted to use this strategy to transfer a huge amount\nof us dollars which they accumulated through inflated contract awards and\nthemoney has been floating in the (c.b.n) since the original beneficiary has\nbeen fully paid, so they wanted to use your account to transfer the surplus\nout of Nigeria. We agreed that once i do this, they would give me\nUS$100,000.00 and give me another us100, 00

In [9]:
import os

os.environ['NLTK_DATA'] = '.'

from nltk import download

download('punkt', download_dir='.')

download('punkt_tab', download_dir='.')

[nltk_data] Downloading package punkt to ....
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to ....
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [10]:
from nltk.tokenize import word_tokenize

In [11]:
from pprint import pprint

In [12]:
elimina_html(contenidos_mensajes_entrenamiento[-21])

'OurRef:CBN/GO/0X012/05 \nDATE:21st JULY 2005\nTEL:234-1-470-7915\nEMAIL ADDRESS:mallamyusuf@centbanks.org \n\nDear Good Friend \n\nAfter a serious thought, I decided to reach you directly and personally\nbecause I do not have anything against you, but your Nigerian partners. \n\nI am the director of wire transfer/telex department of the central bank of\nNigeria, some time in the past your Nigerian partners approached me through a\nfriend of mine who works with one of the ministries here and requested that I\nassist them conclude a money transfer deal and we all agreed. \n\nAccording to them, they wanted to use this strategy to transfer a huge amount\nof us dollars which they accumulated through inflated contract awards and\nthemoney has been floating in the (c.b.n) since the original beneficiary has\nbeen fully paid, so they wanted to use your account to transfer the surplus\nout of Nigeria. We agreed that once i do this, they would give me\nUS$100,000.00 and give me another us100, 00

In [13]:
word_tokenize(elimina_html(contenidos_mensajes_entrenamiento[-21]))

['OurRef',
 ':',
 'CBN/GO/0X012/05',
 'DATE:21st',
 'JULY',
 '2005',
 'TEL:234-1-470-7915',
 'EMAIL',
 'ADDRESS',
 ':',
 'mallamyusuf',
 '@',
 'centbanks.org',
 'Dear',
 'Good',
 'Friend',
 'After',
 'a',
 'serious',
 'thought',
 ',',
 'I',
 'decided',
 'to',
 'reach',
 'you',
 'directly',
 'and',
 'personally',
 'because',
 'I',
 'do',
 'not',
 'have',
 'anything',
 'against',
 'you',
 ',',
 'but',
 'your',
 'Nigerian',
 'partners',
 '.',
 'I',
 'am',
 'the',
 'director',
 'of',
 'wire',
 'transfer/telex',
 'department',
 'of',
 'the',
 'central',
 'bank',
 'of',
 'Nigeria',
 ',',
 'some',
 'time',
 'in',
 'the',
 'past',
 'your',
 'Nigerian',
 'partners',
 'approached',
 'me',
 'through',
 'a',
 'friend',
 'of',
 'mine',
 'who',
 'works',
 'with',
 'one',
 'of',
 'the',
 'ministries',
 'here',
 'and',
 'requested',
 'that',
 'I',
 'assist',
 'them',
 'conclude',
 'a',
 'money',
 'transfer',
 'deal',
 'and',
 'we',
 'all',
 'agreed',
 '.',
 'According',
 'to',
 'them',
 ',',
 'they',


In [14]:
pprint(word_tokenize(elimina_html(contenidos_mensajes_entrenamiento[-21])),
       compact=True)

['OurRef', ':', 'CBN/GO/0X012/05', 'DATE:21st', 'JULY', '2005',
 'TEL:234-1-470-7915', 'EMAIL', 'ADDRESS', ':', 'mallamyusuf', '@',
 'centbanks.org', 'Dear', 'Good', 'Friend', 'After', 'a', 'serious', 'thought',
 ',', 'I', 'decided', 'to', 'reach', 'you', 'directly', 'and', 'personally',
 'because', 'I', 'do', 'not', 'have', 'anything', 'against', 'you', ',', 'but',
 'your', 'Nigerian', 'partners', '.', 'I', 'am', 'the', 'director', 'of',
 'wire', 'transfer/telex', 'department', 'of', 'the', 'central', 'bank', 'of',
 'Nigeria', ',', 'some', 'time', 'in', 'the', 'past', 'your', 'Nigerian',
 'partners', 'approached', 'me', 'through', 'a', 'friend', 'of', 'mine', 'who',
 'works', 'with', 'one', 'of', 'the', 'ministries', 'here', 'and', 'requested',
 'that', 'I', 'assist', 'them', 'conclude', 'a', 'money', 'transfer', 'deal',
 'and', 'we', 'all', 'agreed', '.', 'According', 'to', 'them', ',', 'they',
 'wanted', 'to', 'use', 'this', 'strategy', 'to', 'transfer', 'a', 'huge',
 'amount', 'of'

La eliminación de los caracteres no alfanuméricos se puede realizar mediante expresiones regulares, usando para ello el paquete [re](https://docs.python.org/es/3/library/re.html) de la biblioteca estándar de Python.

In [15]:
import re

In [16]:
def elimina_no_alfanumerico(contenido):
    return [re.sub(r'[^\w]', '', palabra)
            for palabra in contenido
            if re.search(r'\w', palabra)]

In [17]:
def procesa_mensaje(contenido):
    contenido = elimina_html(contenido)
    contenido = word_tokenize(contenido)
    contenido = elimina_no_alfanumerico(contenido)
    return contenido

In [18]:
pprint(procesa_mensaje(contenidos_mensajes_entrenamiento[-21]),
       compact=True)

['OurRef', 'CBNGO0X01205', 'DATE21st', 'JULY', '2005', 'TEL23414707915',
 'EMAIL', 'ADDRESS', 'mallamyusuf', 'centbanksorg', 'Dear', 'Good', 'Friend',
 'After', 'a', 'serious', 'thought', 'I', 'decided', 'to', 'reach', 'you',
 'directly', 'and', 'personally', 'because', 'I', 'do', 'not', 'have',
 'anything', 'against', 'you', 'but', 'your', 'Nigerian', 'partners', 'I', 'am',
 'the', 'director', 'of', 'wire', 'transfertelex', 'department', 'of', 'the',
 'central', 'bank', 'of', 'Nigeria', 'some', 'time', 'in', 'the', 'past',
 'your', 'Nigerian', 'partners', 'approached', 'me', 'through', 'a', 'friend',
 'of', 'mine', 'who', 'works', 'with', 'one', 'of', 'the', 'ministries', 'here',
 'and', 'requested', 'that', 'I', 'assist', 'them', 'conclude', 'a', 'money',
 'transfer', 'deal', 'and', 'we', 'all', 'agreed', 'According', 'to', 'them',
 'they', 'wanted', 'to', 'use', 'this', 'strategy', 'to', 'transfer', 'a',
 'huge', 'amount', 'of', 'us', 'dollars', 'which', 'they', 'accumulated',
 'thr

Debido a la naturaleza de los mensajes no deseados, algunos de ellos pueden confundir a la biblioteca Beautiful Soup, avisando esta de que el mensaje puede tratarse de una URL o de una ruta a un fichero, en lugar de un mensaje de correo electrónico. El código de la siguiente celda filtra ese tipo de avisos.

In [19]:
from bs4 import MarkupResemblesLocatorWarning
import warnings

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

Estamos ya en condiciones de poder construir el filtro de correo electrónico no deseado.

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier

In [21]:
# probar el vectorizador
vectorizador = TfidfVectorizer(analyzer=procesa_mensaje)
vectorizador.fit(contenidos_mensajes_entrenamiento)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",<function pro...x7c1a9f1ebd80>
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer

In [22]:
# total de rasgos
len(vectorizador.get_feature_names_out())

165176

In [25]:
contenidos_mensajes_entrenamiento[-21]

'OurRef:CBN/GO/0X012/05 \nDATE:21st JULY 2005\nTEL:234-1-470-7915\nEMAIL ADDRESS:mallamyusuf@centbanks.org \n\nDear Good Friend \n\nAfter a serious thought, I decided to reach you directly and personally\nbecause I do not have anything against you, but your Nigerian partners. \n\nI am the director of wire transfer/telex department of the central bank of\nNigeria, some time in the past your Nigerian partners approached me through a\nfriend of mine who works with one of the ministries here and requested that I\nassist them conclude a money transfer deal and we all agreed. \n\nAccording to them, they wanted to use this strategy to transfer a huge amount\nof us dollars which they accumulated through inflated contract awards and\nthemoney has been floating in the (c.b.n) since the original beneficiary has\nbeen fully paid, so they wanted to use your account to transfer the surplus\nout of Nigeria. We agreed that once i do this, they would give me\nUS$100,000.00 and give me another us100, 00

In [27]:
# obtener la representacion de 1 documento
tfidf = vectorizador.transform([contenidos_mensajes_entrenamiento[-21]])

In [32]:
# valores internos de tfidf
print(tfidf)
print(type(tfidf))

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 258 stored elements and shape (1, 165176)>
  Coords	Values
  (0, 4)	0.061344582186157896
  (0, 2924)	0.025583845036064576
  (0, 2929)	0.09795934626248219
  (0, 8504)	0.03141164301798313
  (0, 9831)	0.06987781955181108
  (0, 16236)	0.02885715378482451
  (0, 24540)	0.09171579094374219
  (0, 25488)	0.0385780051488376
  (0, 25854)	0.030650865181505796
  (0, 26880)	0.05963274545098519
  (0, 29154)	0.027733245959796475
  (0, 30615)	0.031756604518357136
  (0, 31008)	0.0651564240685496
  (0, 31010)	0.06987781955181108
  (0, 35463)	0.06987781955181108
  (0, 36594)	0.027391915341091477
  (0, 36943)	0.04258173391326585
  (0, 37355)	0.030992339346096544
  (0, 38923)	0.07635442218238718
  (0, 43043)	0.050992237618765146
  (0, 44578)	0.03297074838024415
  (0, 47524)	0.15976799641980743
  (0, 49585)	0.04627084213550367
  (0, 50595)	0.06711598024302888
  (0, 55076)	0.0651564240685496
  :	:
  (0, 155014)	0.10127339899886136
  (0, 155015)	0.0

In [37]:
# vamos a buscar las palabras que corresponden a cada posición, e imprimir el valor de tfidf para cada una de ellas
palabras = vectorizador.get_feature_names_out()

indices_no_cero = tfidf.nonzero()[1]

for i in indices_no_cero:
    print(f'{palabras[i]}: {tfidf[0, i]}')

00000: 0.061344582186157896
100: 0.025583845036064576
10000000: 0.09795934626248219
2005: 0.03141164301798313
23414707915: 0.06987781955181108
50: 0.02885715378482451
ADDRESS: 0.09171579094374219
According: 0.0385780051488376
After: 0.030650865181505796
Approvals: 0.05963274545098519
Best: 0.027733245959796475
But: 0.031756604518357136
CBN: 0.0651564240685496
CBNGO0X01205: 0.06987781955181108
DATE21st: 0.06987781955181108
Dear: 0.027391915341091477
Dept: 0.04258173391326585
Director: 0.030992339346096544
EMAIL: 0.07635442218238718
Friend: 0.050992237618765146
Good: 0.03297074838024415
I: 0.15976799641980743
JULY: 0.04627084213550367
KTT: 0.06711598024302888
Mallam: 0.0651564240685496
Nigeria: 0.09411966893566828
Nigerian: 0.10590358758648885
Now: 0.03124549980938267
OurRef: 0.06987781955181108
Personal: 0.042770083254921785
Regards: 0.02613118652169973
TEL: 0.04659785855709292
TEL23414707915: 0.06987781955181108
They: 0.028487569558871183
TransferTelex: 0.06987781955181108
US: 0.051561

In [38]:
# buscar la aparicion de "00" en contenidos_mensajes_entrenamiento[-21]
# buscar la seccion donde aparezca, imprimir 30 caracteres hacia atras y adelante

mensaje = contenidos_mensajes_entrenamiento[-21]
patron = "00"

# Buscar todas las ocurrencias de "00"
indice = 0
ocurrencias = []

while indice < len(mensaje):
    posicion = mensaje.find(patron, indice)
    if posicion == -1:
        break
    ocurrencias.append(posicion)
    indice = posicion + 1

# Imprimir el contexto de cada ocurrencia
print(f"Se encontraron {len(ocurrencias)} ocurrencias de '{patron}':\n")
for i, pos in enumerate(ocurrencias, 1):
    inicio = max(0, pos - 60)
    fin = min(len(mensaje), pos + len(patron) + 60)
    contexto = mensaje[inicio:fin]
    print(f"Ocurrencia {i} (posición {pos}):")
    print(f"...{contexto}...")
    print()

Se encontraron 14 ocurrencias de '00':

Ocurrencia 1 (posición 40):
...OurRef:CBN/GO/0X012/05 
DATE:21st JULY 2005
TEL:234-1-470-7915
EMAIL ADDRESS:mallamyusuf@centbanks.org...

Ocurrencia 2 (posición 940):
...eria. We agreed that once i do this, they would give me
US$100,000.00 and give me another us100, 000.00 when i released th...

Ocurrencia 3 (posición 943):
...a. We agreed that once i do this, they would give me
US$100,000.00 and give me another us100, 000.00 when i released the f...

Ocurrencia 4 (posición 944):
.... We agreed that once i do this, they would give me
US$100,000.00 and give me another us100, 000.00 when i released the fu...

Ocurrencia 5 (posición 947):
...e agreed that once i do this, they would give me
US$100,000.00 and give me another us100, 000.00 when i released the fund ...

Ocurrencia 6 (posición 973):
...is, they would give me
US$100,000.00 and give me another us100, 000.00 when i released the fund to
your account. When
they...

Ocurrencia 7 (posición 977)

In [39]:
# mostrar las 10 palabras con mayor valor de tfidf

palabras_con_valores = [(palabras[i], tfidf[0, i]) for i in indices_no_cero]

palabras_ordenadas = sorted(palabras_con_valores, key=lambda x: x[1], reverse=True)

print("Las 10 palabras con mayor valor TF-IDF:\n")
for palabra, valor in palabras_ordenadas[:10]:
    print(f'{palabra}: {valor:.6f}')

Las 10 palabras con mayor valor TF-IDF:

to: 0.218289
transfer: 0.215731
they: 0.190056
the: 0.189949
of: 0.187937
I: 0.159768
and: 0.147771
pay: 0.144080
fund: 0.142630
centbanksorg: 0.139756


In [40]:
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=5, metric='cosine'))
])

In [41]:
filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('vectorizador', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [42]:
from sklearn.metrics import recall_score

In [43]:
predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

0.9381387842926304

In [44]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)

print("=" * 60)
print("RESUMEN DE EVALUACIÓN DEL FILTRO ANTISPAM")
print("=" * 60)

# Métricas individuales
accuracy = accuracy_score(clases_mensajes_prueba, predicciones_mensajes_prueba)
precision = precision_score(clases_mensajes_prueba, predicciones_mensajes_prueba)
recall = recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)
f1 = f1_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

print(f"\nMÉTRICAS DE CLASIFICACIÓN:")
print(f"  Exactitud (Accuracy):   {accuracy:.4f}")
print(f"  Precisión (Precision):  {precision:.4f}")
print(f"  Sensibilidad (Recall):  {recall:.4f}")
print(f"  F1-Score:               {f1:.4f}")

# Matriz de confusión
print(f"\nMATRIZ DE CONFUSIÓN:")
cm = confusion_matrix(clases_mensajes_prueba, predicciones_mensajes_prueba)
print(f"                    Predicho: Legítimo  Predicho: Spam")
print(f"  Real: Legítimo           {cm[0][0]:6d}          {cm[0][1]:6d}")
print(f"  Real: Spam               {cm[1][0]:6d}          {cm[1][1]:6d}")

# Reporte de clasificación completo
print(f"\nREPORTE DETALLADO:")
print(classification_report(clases_mensajes_prueba, predicciones_mensajes_prueba, 
                           target_names=['Legítimo', 'No deseado']))

print("=" * 60)

RESUMEN DE EVALUACIÓN DEL FILTRO ANTISPAM

MÉTRICAS DE CLASIFICACIÓN:
  Exactitud (Accuracy):   0.9704
  Precisión (Precision):  0.9776
  Sensibilidad (Recall):  0.9381
  F1-Score:               0.9575

MATRIZ DE CONFUSIÓN:
                    Predicho: Legítimo  Predicho: Spam
  Real: Legítimo             3343              40
  Real: Spam                  115            1744

REPORTE DETALLADO:
              precision    recall  f1-score   support

    Legítimo       0.97      0.99      0.98      3383
  No deseado       0.98      0.94      0.96      1859

    accuracy                           0.97      5242
   macro avg       0.97      0.96      0.97      5242
weighted avg       0.97      0.97      0.97      5242



#### Apartado 1

En este apartado se pide incorporar al procesado de mensajes los siguientes 2 pasos:

* Expandir las contracciones típicas del idioma inglés. Usar para ello el paquete [contractions](https://github.com/kootenpv/contractions).
* Convertir todos los caracteres a minúsculas.

In [45]:
# El paquete contractions posiblemente deba ser instalado, por ejemplo ejecutando en una celda
# !pip install contractions

import contractions

In [46]:
def expande_contracciones(contenido):
    return contractions.fix(contenido)

In [47]:
def convierte_a_minusculas(contenido):
    return contenido.lower()

In [48]:
def procesa_mensaje(contenido):
    contenido = elimina_html(contenido)
    contenido = expande_contracciones(contenido)
    contenido = convierte_a_minusculas(contenido)
    contenido = word_tokenize(contenido)
    contenido = elimina_no_alfanumerico(contenido)
    return contenido

In [49]:
pprint(procesa_mensaje(contenidos_mensajes_entrenamiento[-21]),
       compact=True)

['ourref', 'cbngo0x01205', 'date21st', 'july', '2005', 'tel23414707915',
 'email', 'address', 'mallamyusuf', 'centbanksorg', 'dear', 'good', 'friend',
 'after', 'a', 'serious', 'thought', 'i', 'decided', 'to', 'reach', 'you',
 'directly', 'and', 'personally', 'because', 'i', 'do', 'not', 'have',
 'anything', 'against', 'you', 'but', 'your', 'nigerian', 'partners', 'i', 'am',
 'the', 'director', 'of', 'wire', 'transfertelex', 'department', 'of', 'the',
 'central', 'bank', 'of', 'nigeria', 'some', 'time', 'in', 'the', 'past',
 'your', 'nigerian', 'partners', 'approached', 'me', 'through', 'a', 'friend',
 'of', 'mine', 'who', 'works', 'with', 'one', 'of', 'the', 'ministries', 'here',
 'and', 'requested', 'that', 'i', 'assist', 'them', 'conclude', 'a', 'money',
 'transfer', 'deal', 'and', 'we', 'all', 'agreed', 'according', 'to', 'them',
 'they', 'wanted', 'to', 'use', 'this', 'strategy', 'to', 'transfer', 'a',
 'huge', 'amount', 'of', 'us', 'dollars', 'which', 'they', 'accumulated',
 'thr

In [50]:
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=5, metric='cosine'))
])

In [51]:
filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('vectorizador', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [52]:
predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

0.9429800968262507

In [53]:
predicciones_mensajes_prueba

array([0, 0, 0, ..., 1, 1, 1], shape=(5242,))

#### Apartado 2

Palabras vacías (_stop words_, en inglés) es el nombre que reciben las palabras tales como artículos, pronombres y preposiciones que se considera que no aportan significado para un sistema de procesamiento del lenguaje natural y que, por tanto, deben eliminarse durante las operaciones de preprocesado de texto. El conjunto adecuado de palabras vacías a usar depende del sistema concreto que se esté construyendo, e incluso puede resultar conveniente no hacer uso de esta técnica.

NLTK provee de conjuntos genéricos de palabras vacías para distintos idiomas.

In [54]:
download('stopwords', download_dir='.')

[nltk_data] Downloading package stopwords to ....
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [55]:
from nltk.corpus import stopwords
from nltk.data import path
path.append(".")

In [56]:
palabras_vacias_ingles = stopwords.words('english')
pprint(palabras_vacias_ingles, compact=True)

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an',
 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been',
 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn',
 "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't",
 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from',
 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven',
 "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself',
 "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in',
 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself',
 "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most',
 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not',
 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours',
 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "s

En este apartado se pide incorporar al procesado de mensajes la eliminación de palabras vacías.

In [57]:
def elimina_palabras_vacias(contenido):
    return [palabra
            for palabra in contenido
            if palabra not in palabras_vacias_ingles]

In [58]:
def procesa_mensaje(contenido):
    contenido = elimina_html(contenido)
    contenido = expande_contracciones(contenido)
    contenido = convierte_a_minusculas(contenido)
    contenido = word_tokenize(contenido)
    contenido = elimina_no_alfanumerico(contenido)
    contenido = elimina_palabras_vacias(contenido)
    return contenido

In [ ]:

pprint(procesa_mensaje(contenidos_mensajes_entrenamiento[-21]),
       compact=True)

('OurRef:CBN/GO/0X012/05 \n'
 'DATE:21st JULY 2005\n'
 'TEL:234-1-470-7915\n'
 'EMAIL ADDRESS:mallamyusuf@centbanks.org \n'
 '\n'
 'Dear Good Friend \n'
 '\n'
 'After a serious thought, I decided to reach you directly and personally\n'
 'because I do not have anything against you, but your Nigerian partners. \n'
 '\n'
 'I am the director of wire transfer/telex department of the central bank of\n'
 'Nigeria, some time in the past your Nigerian partners approached me through '
 'a\n'
 'friend of mine who works with one of the ministries here and requested that '
 'I\n'
 'assist them conclude a money transfer deal and we all agreed. \n'
 '\n'
 'According to them, they wanted to use this strategy to transfer a huge '
 'amount\n'
 'of us dollars which they accumulated through inflated contract awards and\n'
 'themoney has been floating in the (c.b.n) since the original beneficiary '
 'has\n'
 'been fully paid, so they wanted to use your account to transfer the surplus\n'
 'out of Nigeria. W

In [61]:
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=5, metric='cosine'))
])

In [62]:
filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('vectorizador', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [63]:
predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

0.9467455621301775

#### Apartado 3

Por razones gramaticales, en un documento de texto van a aparecer con seguridad diferentes formas de una palabra, como organizar, organiza y organizando. Además, existen familias de palabras relacionadas derivativamente con significados similares, como democracia, democrático y democratización. En muchas situaciones, parece que sería útil reducir esos conjuntos de palabras a una raíz común. Para ello se suelen usar los procedimientos de _stemming_ y lematización.

_Stemming_ generalmente se refiere a un proceso heurístico rudimentario que corta los extremos de las palabras con la esperanza de lograr el objetivo correctamente la mayor parte del tiempo y, a menudo, incluye la eliminación de afijos derivativos. La lematización generalmente se refiere a hacer las cosas correctamente con el uso de un vocabulario y análisis morfológico de las palabras, normalmente con el objetivo de eliminar únicamente las terminaciones flexivas y devolver la forma base o de diccionario de una palabra, lo que se conoce como lema.

NLTK provee de varios algoritmos de _stemming_ y lematización. En este apartado se pide incorporar al procesado de mensajes el procedimiento de _stemming_ mediante el [algoritmo de Lancaster](https://www.nltk.org/api/nltk.stem.lancaster.html).

In [64]:
from nltk.stem.lancaster import LancasterStemmer

In [65]:
def reduce_a_raiz(contenido):
    reductor = LancasterStemmer()
    return [reductor.stem(palabra)
            for palabra in contenido]

In [66]:
def procesa_mensaje(contenido):
    contenido = elimina_html(contenido)
    contenido = expande_contracciones(contenido)
    contenido = convierte_a_minusculas(contenido)
    contenido = word_tokenize(contenido)
    contenido = elimina_no_alfanumerico(contenido)
    contenido = elimina_palabras_vacias(contenido)
    contenido = reduce_a_raiz(contenido)
    return contenido

In [67]:
pprint(procesa_mensaje(contenidos_mensajes_entrenamiento[-21]),
       compact=True)

['ourref', 'cbngo0x01205', 'date21st', 'july', '2005', 'tel23414707915',
 'email', 'address', 'mallamyusuf', 'centbanksorg', 'dear', 'good', 'friend',
 'sery', 'thought', 'decid', 'reach', 'direct', 'person', 'anyth', 'nig',
 'partn', 'direct', 'wir', 'transfertelex', 'depart', 'cent', 'bank', 'niger',
 'tim', 'past', 'nig', 'partn', 'approach', 'friend', 'min', 'work', 'on',
 'min', 'request', 'assist', 'conclud', 'money', 'transf', 'deal', 'agree',
 'accord', 'want', 'us', 'strategy', 'transf', 'hug', 'amount', 'us', 'doll',
 'accum', 'infl', 'contract', 'award', 'themoney', 'flo', 'cbn', 'sint',
 'origin', 'beneficy', 'ful', 'paid', 'want', 'us', 'account', 'transf',
 'surpl', 'niger', 'agree', 'would', 'giv', 'us', '10000000', 'giv', 'anoth',
 'us100', '00000', 'releas', 'fund', 'account', 'saw', 'don', 'nam', 'approv',
 'among', 'list', 'paid', 'instead', 'giv', 'agree', 'deposit', 'us',
 '10000000', 'start', 'avoid', 'resort', 'threats', 'immedy', 'delet', 'transf',
 'cod', 'fund

In [68]:
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=4, metric='cosine'))
])

In [69]:
filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('vectorizador', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [70]:
predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

0.92845615922539

### Ejercicio 2

En el cuaderno NLTK.ipynb se ha construido un sistema de predicción de texto en español basado en modelos de $n$-gramas. Estos modelos se han entrenado a partir de un corpus de textos en español que se ha usado en bruto. El objetivo de este ejercicio es recrear la construcción del sistema de predicción de texto, pero usando una versión normalizada del corpus.

#### Apartado 1

En este apartado se pide:

1. Leer el corpus guardado en el fichero `Texto predictivo/corpus_InfoLibros_parcial.txt` y dividirlo en un corpus de entrenamiento y un corpus de prueba.
2. Construir modelos unigramas, bigramas y trigramas, con y sin suavizado, a partir del corpus de entrenamiento normalizado convirtiendo todas las palabras a minúsculas.
3. Seleccionar el modelo con menor perplejidad sobre el corpus de prueba normalizado convirtiendo todas las palabras a minúsculas.

In [71]:
# Nos aseguramos de haber descargado el tokenizador

from nltk import download

download('punkt', download_dir='.')

[nltk_data] Downloading package punkt to ....
[nltk_data]   Package punkt is already up-to-date!


True

In [72]:
from nltk.corpus.reader.plaintext import PlaintextCorpusReader
from nltk.data import load

In [73]:
corpus_InfoLibros = PlaintextCorpusReader(
    root='Texto predictivo',
    fileids=['corpus_InfoLibros_parcial.txt'],
    encoding='utf8',
    sent_tokenizer=load('tokenizers/punkt/spanish.pickle')
)

In [74]:
total_frases = len(corpus_InfoLibros.sents())
total_frases

1008664

In [75]:
total_frases_entrenamiento = int(total_frases * .8)
total_frases_entrenamiento

806931

In [76]:
corpus_entrenamiento = corpus_InfoLibros.sents()[:total_frases_entrenamiento]

In [77]:
corpus_prueba = corpus_InfoLibros.sents()[total_frases_entrenamiento:]

In [78]:
from nltk.lm.vocabulary import Vocabulary
from nltk.lm.preprocessing import flatten

In [79]:
vocabulario_palabras = Vocabulary(
    (palabra.lower()
     for palabra in flatten(corpus_entrenamiento)),  # lista de todas las palabras
    unk_cutoff=50  # mínimo número de ocurrencias
)

In [80]:
vocabulario_palabras.lookup('hola')

'hola'

In [81]:
inicio_frase = '<s>'
fin_frase = '</s>'
vocabulario_palabras.update({inicio_frase: 50, fin_frase: 50})

In [82]:
def delimita_frase(frase, n):
    return (['<s>'] * (n - 1) +
            [palabra.lower() for palabra in frase] +
            ['</s>'])

In [83]:
from pprint import pprint

In [84]:
primera_frase_entrenamiento = corpus_entrenamiento[0]
pprint(delimita_frase(primera_frase_entrenamiento, 1),
       compact=True)
pprint(delimita_frase(primera_frase_entrenamiento, 2),
       compact=True)
pprint(delimita_frase(primera_frase_entrenamiento, 3),
       compact=True)

['venga', 'usted', 'aquí', ',', 'querida', 'elena', '-', 'dijo', 'ana',
 'pavlovna', 'a', 'la', 'bella', 'princesa', ',', 'que', ',', 'sentada', 'un',
 'poco', 'más', 'lejos', ',', 'formaba', 'el', 'centro', 'del', 'otro', 'grupo',
 '.', '</s>']
['<s>', 'venga', 'usted', 'aquí', ',', 'querida', 'elena', '-', 'dijo', 'ana',
 'pavlovna', 'a', 'la', 'bella', 'princesa', ',', 'que', ',', 'sentada', 'un',
 'poco', 'más', 'lejos', ',', 'formaba', 'el', 'centro', 'del', 'otro', 'grupo',
 '.', '</s>']
['<s>', '<s>', 'venga', 'usted', 'aquí', ',', 'querida', 'elena', '-', 'dijo',
 'ana', 'pavlovna', 'a', 'la', 'bella', 'princesa', ',', 'que', ',', 'sentada',
 'un', 'poco', 'más', 'lejos', ',', 'formaba', 'el', 'centro', 'del', 'otro',
 'grupo', '.', '</s>']


In [85]:
from nltk.util import ngrams, bigrams, trigrams
from nltk.lm import MLE, Laplace

In [86]:
for ii in ngrams(delimita_frase(corpus_entrenamiento[0], 1), n=1):
    print(ii)

('venga',)
('usted',)
('aquí',)
(',',)
('querida',)
('elena',)
('-',)
('dijo',)
('ana',)
('pavlovna',)
('a',)
('la',)
('bella',)
('princesa',)
(',',)
('que',)
(',',)
('sentada',)
('un',)
('poco',)
('más',)
('lejos',)
(',',)
('formaba',)
('el',)
('centro',)
('del',)
('otro',)
('grupo',)
('.',)
('</s>',)


In [87]:
aa = (ngrams(delimita_frase(frase, 1), n=1) for frase in corpus_entrenamiento)

In [88]:
aaa = next(aa)

In [89]:
next(aaa)

('venga',)

In [90]:
modelo_unigrama_MLE = MLE(1, vocabulary=vocabulario_palabras)
modelo_unigrama_MLE.fit(ngrams(delimita_frase(frase, 1), n=1)
                        for frase in corpus_entrenamiento)
modelo_unigrama_MLE.perplexity(flatten(ngrams(delimita_frase(frase, 1), n=1)
                                       for frase in corpus_prueba))

541.9509892055994

In [91]:
modelo_unigrama_Laplace = Laplace(1, vocabulary=vocabulario_palabras)
modelo_unigrama_Laplace.fit(ngrams(delimita_frase(frase, 1), n=1)
                            for frase in corpus_entrenamiento)
modelo_unigrama_Laplace.perplexity(flatten(ngrams(delimita_frase(frase, 1), n=1)
                                           for frase in corpus_prueba))

541.9669927536652

In [92]:
modelo_bigrama_MLE = MLE(2, vocabulary=vocabulario_palabras)
modelo_bigrama_MLE.fit(bigrams(delimita_frase(frase, 2))
                       for frase in corpus_entrenamiento)
modelo_bigrama_MLE.perplexity(flatten(bigrams(delimita_frase(frase, 2))
                                      for frase in corpus_prueba))

inf

In [93]:
modelo_bigrama_Laplace = Laplace(2, vocabulary=vocabulario_palabras)
modelo_bigrama_Laplace.fit(bigrams(delimita_frase(frase, 2))
                           for frase in corpus_entrenamiento)
modelo_bigrama_Laplace.perplexity(flatten(bigrams(delimita_frase(frase, 2))
                                          for frase in corpus_prueba))

367.182902737889

In [ ]:
modelo_trigrama_MLE = MLE(3, vocabulary=vocabulario_palabras)
modelo_trigrama_MLE.fit(trigrams(delimita_frase(frase, 3))
                        for frase in corpus_entrenamiento)
modelo_trigrama_MLE.perplexity(flatten(trigrams(delimita_frase(frase, 3))
                                       for frase in corpus_prueba))

In [ ]:
modelo_trigrama_Laplace = Laplace(3, vocabulary=vocabulario_palabras)
modelo_trigrama_Laplace.fit(trigrams(delimita_frase(frase, 3))
                            for frase in corpus_entrenamiento)
modelo_trigrama_Laplace.perplexity(flatten(trigrams(delimita_frase(frase, 3))
                                           for frase in corpus_prueba))

#### Apartado 2

Definir una función `predice_palabras` que prediga, a partir de las palabras anteriores y de las letras de la palabra ya escritas, qué palabra se pretende escribir. La función debe actuar como sigue:

* Si todas las letras del prefijo escrito están en minúsculas, entonces debe predecir palabras en minúsculas.
* Si todas las letras del prefijo escrito están en mayúsculas, entonces debe predecir palabras en mayúsculas.
* Si el prefijo escrito mezcla letras en minúsculas y en mayúsculas, entonces:
  * Si la primera letra del prefijo está en minúsculas, entonces debe predecir palabras en minúsculas.
  * Si la primera letra del prefijo está en mayúsculas, entonces debe predecir palabras con la primera letra en mayúsculas y el resto en minúsculas.

In [ ]:
def predice_palabras(prefijo, contexto, numero_palabras, modelo):
    def puntua_palabra(palabra):
        return modelo.score(
            palabra,
            tuple(palabra_anterior.lower()
                  for palabra_anterior in contexto)
        )
    palabras_candidatas = filter(
        lambda palabra: palabra.startswith(prefijo.lower()),
        vocabulario_palabras
    )
    palabras_candidatas_ordenadas = sorted(palabras_candidatas,
                                           key=puntua_palabra,
                                           reverse=True)
    palabras_predichas = palabras_candidatas_ordenadas[:numero_palabras]
    if prefijo.islower():
        return palabras_predichas
    elif prefijo.isupper():
        return [palabra_predicha.upper()
                for palabra_predicha in palabras_predichas]
    elif prefijo[0].islower():
        return palabras_predichas
    else:
        return [palabra_predicha.capitalize()
                for palabra_predicha in palabras_predichas]

In [ ]:
predice_palabras('nat', ('Lenguaje',), 5, modelo_bigrama_Laplace)

In [ ]:
predice_palabras('NAT', ('Lenguaje',), 5, modelo_bigrama_Laplace)

In [ ]:
predice_palabras('nAt', ('Lenguaje',), 5, modelo_bigrama_Laplace)

In [ ]:
predice_palabras('NaT', ('Lenguaje',), 5, modelo_bigrama_Laplace)